# Week 1 · Day 5 · Task 2: Exploratory Data Analysis
Input: the cleaned dataset from Task 1 (`task1_data_cleaning/data/clean_data.csv`).
The analysis functions are in `src/analysis.py`. The business questions show their pandas code directly in the cells below.

In [1]:
from pathlib import Path
import sys

# Find this task's folder whether the notebook runs from notebooks/, the task folder or the repo root.
TASK = "task2_exploratory_analysis"
HERE = Path.cwd().resolve()
candidates = [p for base in [HERE, *HERE.parents]
              for p in (base, base / "week1" / "day5" / TASK)]
PROJECT_DIR = next(p for p in candidates if (p / "src" / "analysis.py").is_file())
sys.path.insert(0, str(PROJECT_DIR / "src"))
print("Project folder:", PROJECT_DIR.name)

import pandas as pd
import analysis as a
pd.set_option("display.width", 120)

Project folder: task2_exploratory_analysis


In [2]:
df = a.load_clean_data()        # also saves a snapshot copy in data/clean_data.csv
print(df.shape)
df

(8, 6)


,id,name,department,salary,city,salary_outlier
0,1,Arun,Sales,65000,Chennai,False
1,2,Priya,HR,45000,Bangalore,False
2,3,Karthik,IT,58000,Chennai,False
3,4,Divya,IT,75000,Mumbai,False
4,5,Rohan,Finance,81000,Chennai,False
5,6,Meena,Sales,52000,Bangalore,False
6,7,Suresh,IT,68000,Hyderabad,False
7,8,Anitha,HR,48000,Mumbai,False


## A. Summary statistics
`id` is an identifier and `salary_outlier` is a True/False flag, so neither is a measurement.
**`salary` is the only numeric measure in the dataset.**

In [3]:
print("Measure columns:", a.measure_columns(df))
stats = a.summary_statistics(df)
stats.to_csv(a.REPORTS_DIR / "summary_statistics.csv")
stats

Measure columns: ['salary']


,count,mean,median,min,max,std
column,,,,,,
salary,8.0,61500.0,61500.0,45000.0,81000.0,12950.46


Mean and median are both **61,500**, so salaries are spread evenly around the centre. The standard deviation is about 12,950.

## B. Group analysis: `groupby().agg()`

In [4]:
a.group_summary(df, "city")

,employees,avg_salary,median_salary,min_salary,max_salary,total_salary
city,,,,,,
Chennai,3,68000.0,65000.0,58000,81000,204000
Hyderabad,1,68000.0,68000.0,68000,68000,68000
Mumbai,2,61500.0,61500.0,48000,75000,123000
Bangalore,2,48500.0,48500.0,45000,52000,97000


In [5]:
a.group_summary(df, "department")

,employees,avg_salary,median_salary,min_salary,max_salary,total_salary
department,,,,,,
Finance,1,81000.0,81000.0,81000,81000,81000
IT,3,67000.0,68000.0,58000,75000,201000
Sales,2,58500.0,58500.0,52000,65000,117000
HR,2,46500.0,46500.0,45000,48000,93000


* **City:** Chennai (3 employees) and Hyderabad (1) share the highest average, 68,000. Bangalore is lowest at 48,500.
* **Department:** Finance is highest (81,000) but has only one employee. IT has the most employees (3) with an average of 67,000. HR is lowest (46,500).
* With only 1 to 3 employees per group, one salary can change a group's average a lot.

## C. Correlation analysis

In [6]:
corr = a.correlation_matrix(df)          # = df.drop(columns=flag columns).corr(numeric_only=True)
corr.round(4).to_csv(a.REPORTS_DIR / "correlation_matrix.csv")
corr.round(4)

,id,salary
id,1.000,-0.036
salary,-0.036,1.000


In [7]:
a.top_correlations(corr, n=3)            # different columns only, each pair once

,column_1,column_2,correlation,abs_correlation
0,id,salary,-0.036027,0.036027


**Limitation (no relationships are invented here):** `df.corr(numeric_only=True)` finds two numeric columns, `id` and `salary`.
That gives **only one pair**, so a "top three" list cannot exist. The single value (≈ −0.04) is also **not a real relationship**,
because `id` is just a row number and does not measure anything about the employee.
A useful correlation analysis needs at least two real measures, for example age, years of experience or performance rating.

Remember: **correlation does not prove causation.** Even a strong correlation would only show that two values move together,
not that one causes the other.

## D. Business questions (reused from Day 4 Task 2)

### Q1. What is the average salary across all employees?

In [8]:
df["salary"].agg(["mean", "median", "min", "max"])

mean      61500.0
median    61500.0
min       45000.0
max       81000.0
Name: salary, dtype: float64

**Interpretation:** a typical employee earns about **61,500** (mean and median are the same). The range is 45,000 to 81,000, so the highest salary is 1.8 times the lowest.

### Q2. Which department has the highest average salary?

In [9]:
df.groupby("department", observed=True)["salary"].agg(["mean", "count"]).sort_values("mean", ascending=False)

,mean,count
department,,
Finance,81000.0,1
IT,67000.0,3
Sales,58500.0,2
HR,46500.0,2


**Interpretation:** **Finance** has the highest average (81,000), but that is one person. Among departments with more than one employee, **IT** leads (67,000). HR is lowest (46,500).

### Q3. Which city has the highest average salary, and how does salary vary between cities?

In [10]:
df.groupby("city", observed=True)["salary"].agg(["mean", "min", "max", "count"]).sort_values("mean", ascending=False)

,mean,min,max,count
city,,,,
Chennai,68000.0,58000,81000,3
Hyderabad,68000.0,68000,68000,1
Mumbai,61500.0,48000,75000,2
Bangalore,48500.0,45000,52000,2


**Interpretation:** **Chennai and Hyderabad tie at 68,000**. Bangalore is lowest (48,500), a gap of 19,500. Without the city cleaning in Task 1, Chennai would have been split into three groups with averages of 65,000, 58,000 and 81,000.

### Q4. Which employee has the highest salary?

In [11]:
df.loc[[df["salary"].idxmax()], ["id", "name", "department", "city", "salary"]]

,id,name,department,city,salary
4,5,Rohan,Finance,Chennai,81000


**Interpretation:** **Rohan** (id 5, Finance, Chennai) earns the most, 81,000. This is 19,500 above the company average.

### Q5. How are employees distributed across departments and cities?

In [12]:
pd.crosstab(df["city"], df["department"], margins=True, margins_name="Total")

department,Finance,HR,IT,Sales,Total
city,,,,,
Bangalore,0,1,0,1,2
Chennai,1,0,1,1,3
Hyderabad,0,0,1,0,1
Mumbai,0,1,1,0,2
Total,1,2,3,2,8


**Interpretation:** IT is the largest department (3) and Chennai the largest city (3). Most city/department combinations have 0 or 1 employees.

In [13]:
pd.DataFrame(a.business_questions(df))     # the same answers, as calculated by analysis.py

,question,answer
0,Q1. What is the average salary across all empl...,"Mean 61,500; median 61,500; range 45,000-81,000."
1,Q2. Which department has the highest average s...,"Finance (81,000, 1 employee(s))."
2,Q3. Which city has the highest average salary?,"Chennai and Hyderabad (68,000); lowest is Bang..."
3,Q4. Which employee has the highest salary?,"Rohan (id 5, Finance, Chennai): 81,000."
4,Q5. How are employees distributed across depar...,Largest department: IT (3); largest city: Chen...


## E. Merge another dataset
The employee data **has** a `department` column, so we use a small department budget lookup table.
⚠️ **The budget values in `data/department_budget.csv` are ILLUSTRATIVE.** They were written for this exercise and are not real company data.
Every row says so in the `value_type` column.

In [14]:
print("Original dataset:")
display(df[["id", "name", "department", "salary"]])
budget = pd.read_csv(a.BUDGET_FILE)
print("Lookup / budget dataset:")
budget

Original dataset:


,id,name,department,salary
0,1,Arun,Sales,65000
1,2,Priya,HR,45000
2,3,Karthik,IT,58000
3,4,Divya,IT,75000
4,5,Rohan,Finance,81000
5,6,Meena,Sales,52000
6,7,Suresh,IT,68000
7,8,Anitha,HR,48000


Lookup / budget dataset:


,department,annual_budget,budget_owner,value_type
0,Sales,600000,Head of Sales,ILLUSTRATIVE - not real company data
1,HR,300000,Head of HR,ILLUSTRATIVE - not real company data
2,IT,900000,Head of IT,ILLUSTRATIVE - not real company data
3,Finance,400000,Head of Finance,ILLUSTRATIVE - not real company data


In [15]:
# pd.merge with a LEFT join on "department" (done inside a.merge_budget):
#   - left join keeps every employee, even if their department is missing from the lookup
#   - validate="many_to_one": many employees -> one budget row per department
#   - indicator=True adds _merge, which shows "both" when an employee found a matching budget row
merged = a.merge_budget(df, budget)
print("Unmatched employees:", (merged["_merge"] != "both").sum())
merged

Unmatched employees: 0


,id,name,department,salary,city,salary_outlier,annual_budget,budget_owner,value_type,_merge
0,1,Arun,Sales,65000,Chennai,False,600000,Head of Sales,ILLUSTRATIVE - not real company data,both
1,2,Priya,HR,45000,Bangalore,False,300000,Head of HR,ILLUSTRATIVE - not real company data,both
2,3,Karthik,IT,58000,Chennai,False,900000,Head of IT,ILLUSTRATIVE - not real company data,both
3,4,Divya,IT,75000,Mumbai,False,900000,Head of IT,ILLUSTRATIVE - not real company data,both
4,5,Rohan,Finance,81000,Chennai,False,400000,Head of Finance,ILLUSTRATIVE - not real company data,both
5,6,Meena,Sales,52000,Bangalore,False,600000,Head of Sales,ILLUSTRATIVE - not real company data,both
6,7,Suresh,IT,68000,Hyderabad,False,900000,Head of IT,ILLUSTRATIVE - not real company data,both
7,8,Anitha,HR,48000,Mumbai,False,300000,Head of HR,ILLUSTRATIVE - not real company data,both


In [16]:
a.budget_by_department(merged)

,employees,total_salary,annual_budget,salary_pct_of_budget
department,,,,
Finance,1,81000,400000,20.2
HR,2,93000,300000,31.0
IT,3,201000,900000,22.3
Sales,2,117000,600000,19.5


**Merge key:** `department`, which exists in both tables. Each department appears exactly once in the lookup.
**Join type:** left. All 8 employees are kept and all 8 found a budget row (`_merge = both`).
**Example use:** total salary as a share of the (illustrative) budget. HR is highest at 31.0%.
These percentages show how a merge enables new analysis. They are **not** real findings, and the salary period (monthly or annual) is not documented.

## F. Bonus: Pivot table (mean salary, city × department)

In [17]:
pivot = a.salary_pivot(df)     # pd.pivot_table(df, index="city", columns="department", values="salary", aggfunc="mean")
pivot.to_csv(a.REPORTS_DIR / "pivot_table.csv")
pivot

department,Finance,HR,IT,Sales
city,,,,
Bangalore,NaN,45000.0,NaN,52000.0
Chennai,81000.0,NaN,58000.0,65000.0
Hyderabad,NaN,NaN,68000.0,NaN
Mumbai,NaN,48000.0,75000.0,NaN


Empty cells (NaN) mean **no employee** exists for that city/department pair. They are left empty on purpose: filling them with 0 would wrongly suggest a salary of 0.
IT is the only department in three cities (Chennai 58,000, Hyderabad 68,000, Mumbai 75,000).

## Conclusions and limitations
* Average salary 61,500. Finance and IT pay the most, HR the least. Chennai and Hyderabad have the highest city average.
* Correlation analysis is not meaningful with one numeric measure. This limitation is stated rather than hidden.
* **8 employees is far too few for business decisions.** The notebook demonstrates the method and the code works for a larger file with the same columns.
* Budget values are illustrative. The salary currency and period are unknown.